# Exploring the Embedding Space of the Clay v1.5 Encoder

This notebook demonstrates how to generate and visualize embeddings from the
Clay v1.5 foundation model. We will:

1. Fetch Sentinel-2 imagery from the Element84 STAC catalog (multiple dates)
2. Load the Clay v1.5 checkpoint
3. Prepare datacubes with normalized pixels, wavelengths, and metadata
4. Run the encoder to produce patch-level embeddings
5. Visualize embedding dimensions as spatial feature maps
6. Compare how a single embedding dimension varies across dates

The encoder outputs a tensor of shape `[B, 1 + 1024, 1024]` where:
- The first token is the **CLS token** (global image embedding)
- The remaining 1024 tokens are **patch embeddings** on a 32x32 spatial grid
  (256 pixels / 8 pixel patch size = 32)
- Each token has 1024 embedding dimensions

**Requirements**: `pip install claymodel stackstac pystac-client geopandas matplotlib einops`

In [ ]:
import math
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
import stackstac
import torch
from einops import rearrange
from rasterio.enums import Resampling
from shapely import Point
from torchvision.transforms import v2

from claymodel.module import ClayMAEModule

## Fetch Sentinel-2 data from STAC

We use the same location and date range as the wall-to-wall tutorial: Monchique,
Portugal during a 2018 forest fire. This gives us imagery across a dramatic
landscape change, which is ideal for exploring how embeddings respond.

In [ ]:
# Point over Monchique, Portugal (forest fire site)
lat, lon = 37.30939, -8.57207

# Sentinel-2 bands that match metadata.yaml band_order
S2_BANDS = [
    "blue",
    "green",
    "red",
    "rededge1",
    "rededge2",
    "rededge3",
    "nir",
    "nir08",
    "swir16",
    "swir22",
]

# Search the Element84 STAC catalog
STAC_API = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(STAC_API)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2018-07-01/2018-09-01",
    bbox=(lon - 1e-5, lat - 1e-5, lon + 1e-5, lat + 1e-5),
    max_items=100,
    query={"eo:cloud_cover": {"lt": 80}},
)
all_items = search.item_collection()

# Deduplicate by date
items, dates = [], []
for item in all_items:
    if item.datetime.date() not in dates:
        items.append(item)
        dates.append(item.datetime.date())

print(f"Found {len(items)} unique-date items")

In [ ]:
# Create 256x256 chip bounds in the image projection
epsg = int(items[0].properties["proj:code"].replace("EPSG:", ""))

poidf = gpd.GeoDataFrame(
    pd.DataFrame(), crs="EPSG:4326", geometry=[Point(lon, lat)]
).to_crs(epsg)
coords = poidf.iloc[0].geometry.coords[0]

size, gsd = 256, 10
bounds = (
    coords[0] - (size * gsd) // 2,
    coords[1] - (size * gsd) // 2,
    coords[0] + (size * gsd) // 2,
    coords[1] + (size * gsd) // 2,
)

# Download the imagery stack
stack = stackstac.stack(
    items,
    bounds=bounds,
    snap_bounds=False,
    epsg=epsg,
    resolution=gsd,
    dtype="float64",
    rescale=False,
    fill_value=0,
    assets=S2_BANDS,
    resampling=Resampling.nearest,
)
stack = stack.compute()
print(f"Stack shape: {stack.shape}  (time, band, y, x)")

## Visualize RGB for a few dates

The time series spans the 2018 Monchique forest fire. Early dates show green
forest, some dates have clouds, and later dates show the burn scar.

In [ ]:
# Show RGB composites for each date
n_dates = stack.shape[0]
cols = min(n_dates, 6)
rows = math.ceil(n_dates / cols)

fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes = np.atleast_2d(axes)

for idx in range(rows * cols):
    ax = axes.flatten()[idx]
    if idx < n_dates:
        rgb = stack.isel(time=idx).sel(band=["red", "green", "blue"]).values
        ax.imshow(np.clip(rgb.transpose(1, 2, 0) / 3000, 0, 1))
        date_str = str(stack.time.values[idx])[:10]
        ax.set_title(date_str, fontsize=8)
    ax.set_axis_off()

plt.suptitle("Sentinel-2 RGB time series", fontsize=12)
plt.tight_layout()

## Load the Clay v1.5 model

We load the checkpoint with `mask_ratio=0.0` and `shuffle=False` so the
encoder returns all 1024 patches in spatial order (no masking).

In [ ]:
# Find or download checkpoint
ckpt = "clay-v1.5.ckpt"
if not os.path.exists(ckpt):
    ckpt = "../../clay-v1.5.ckpt"
if not os.path.exists(ckpt):
    print("Downloading checkpoint...")
    os.system(
        "wget -q https://huggingface.co/made-with-clay/Clay/resolve/main/"
        "v1.5/clay-v1.5.ckpt -O clay-v1.5.ckpt"
    )
    ckpt = "clay-v1.5.ckpt"

In [ ]:
from claymodel.api import _bundled_metadata_path, load_metadata

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
torch.set_default_device(device)

metadata = load_metadata()

model = ClayMAEModule.load_from_checkpoint(
    ckpt,
    metadata_path=_bundled_metadata_path(),
    mask_ratio=0.0,
    shuffle=False,
)
model.eval()
model = model.to(device)
print(f"Model loaded on {device}")

## Prepare datacubes

The Clay encoder expects a dictionary with:
- `pixels`: `[B, C, H, W]` z-score normalized using per-band statistics from `metadata.yaml`
- `waves`: central wavelength per band (lets the model adapt to any sensor)
- `gsd`: ground sampling distance in meters
- `time`: `[B, 4]` cyclic encoding of week-of-year and hour-of-day
- `latlon`: `[B, 4]` cyclic encoding of latitude and longitude

In [ ]:
def normalize_timestamp(date):
    """Encode date as cyclic (sin, cos) features for week and hour."""
    week = date.isocalendar().week * 2 * np.pi / 52
    hour = date.hour * 2 * np.pi / 24
    return (math.sin(week), math.cos(week)), (math.sin(hour), math.cos(hour))


def normalize_latlon(lat, lon):
    """Encode lat/lon as (sin, cos) features."""
    lat = lat * np.pi / 180
    lon = lon * np.pi / 180
    return (math.sin(lat), math.cos(lat)), (math.sin(lon), math.cos(lon))

In [ ]:
# Extract normalization stats and wavelengths from metadata
platform = "sentinel-2-l2a"

mean = [metadata[platform].bands.mean[b] for b in stack.band.values]
std = [metadata[platform].bands.std[b] for b in stack.band.values]
waves = [metadata[platform].bands.wavelength[b] for b in stack.band.values]

transform = v2.Compose([v2.Normalize(mean=mean, std=std)])

# Normalize pixels
pixels = torch.from_numpy(stack.values.astype(np.float32))
pixels = transform(pixels)

# Encode timestamps
datetimes = stack.time.values.astype("datetime64[s]").tolist()
times = [normalize_timestamp(dt) for dt in datetimes]
week_norm = [t[0] for t in times]
hour_norm = [t[1] for t in times]

# Encode location (same for all dates)
latlons = [normalize_latlon(lat, lon)] * len(times)
lat_norm = [ll[0] for ll in latlons]
lon_norm = [ll[1] for ll in latlons]

# Build the datacube dictionary
datacube = {
    "pixels": pixels.to(device),
    "time": torch.tensor(
        np.hstack((week_norm, hour_norm)), dtype=torch.float32, device=device
    ),
    "latlon": torch.tensor(
        np.hstack((lat_norm, lon_norm)), dtype=torch.float32, device=device
    ),
    "gsd": torch.tensor(metadata[platform].gsd, device=device),
    "waves": torch.tensor(waves, device=device),
}

print(f"pixels:  {datacube['pixels'].shape}")
print(f"time:    {datacube['time'].shape}")
print(f"latlon:  {datacube['latlon'].shape}")
print(f"waves:   {datacube['waves'].shape}")
print(f"gsd:     {datacube['gsd']}")

## Run the encoder

With `mask_ratio=0.0`, the encoder returns all patches. The output shape is
`[B, 1 + 1024, 1024]` where B is the number of dates. The first token at
position 0 is the CLS token (global image embedding).

In [ ]:
with torch.no_grad():
    unmsk_patch, unmsk_idx, msk_idx, msk_matrix = model.model.encoder(datacube)

print(f"Encoder output shape: {unmsk_patch.shape}")
print(f"  Batch size (dates):    {unmsk_patch.shape[0]}")
print(f"  Tokens per image:      {unmsk_patch.shape[1]}  (1 CLS + 1024 patches)")
print(f"  Embedding dimension:   {unmsk_patch.shape[2]}")

## Visualize patch embeddings as spatial feature maps

We rearrange the 1024 patch tokens back into a 32x32 spatial grid. Each
embedding dimension becomes a "feature map" that highlights different aspects
of the scene -- edges, textures, spectral properties, land cover boundaries, etc.

In [ ]:
# Rearrange patch embeddings (skip CLS at index 0) into spatial grid
# From [B, 1024, D] -> [B, D, 32, 32]
spatial_embeds = rearrange(
    unmsk_patch[:, 1:, :].cpu().numpy(),
    "b (h w) d -> b d h w",
    h=32,
    w=32,
)
print(f"Spatial embeddings shape: {spatial_embeds.shape}  [batch, embed_dim, 32, 32]")

In [ ]:
# Plot the first 256 embedding dimensions for the first date as a 16x16 grid
sample = spatial_embeds[0]  # First date
fig, axs = plt.subplots(16, 16, figsize=(20, 20))

for idx, ax in enumerate(axs.flatten()):
    ax.imshow(sample[idx], cmap="bwr", interpolation="nearest")
    ax.set_axis_off()
    ax.set_title(str(idx), fontsize=5, pad=1)

date_str = str(stack.time.values[0])[:10]
plt.suptitle(
    f"First 256 embedding dimensions as spatial features ({date_str})",
    y=1.01,
    fontsize=14,
)
plt.tight_layout()

## Compare one embedding dimension across all dates

Here we pick a single embedding dimension and show it side by side with the RGB
input for every date. This reveals how a specific learned feature responds to
real changes in the landscape (clouds, fire, regrowth).

In [ ]:
# Pick an embedding dimension to inspect
EMBED_DIM = 42

fig, axes = plt.subplots(n_dates, 2, figsize=(8, 3 * n_dates))

# Use a shared color range across dates for consistent comparison
vmin = spatial_embeds[:, EMBED_DIM].min()
vmax = spatial_embeds[:, EMBED_DIM].max()

for i in range(n_dates):
    date_str = str(stack.time.values[i])[:10]

    # Left: RGB
    rgb = stack.isel(time=i).sel(band=["red", "green", "blue"]).values
    axes[i, 0].imshow(np.clip(rgb.transpose(1, 2, 0) / 3000, 0, 1))
    axes[i, 0].set_title(f"RGB  {date_str}", fontsize=9)
    axes[i, 0].set_axis_off()

    # Right: embedding dimension
    im = axes[i, 1].imshow(
        spatial_embeds[i, EMBED_DIM],
        cmap="bwr",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    axes[i, 1].set_title(f"Dim {EMBED_DIM}  {date_str}", fontsize=9)
    axes[i, 1].set_axis_off()

plt.suptitle(
    f"RGB vs embedding dimension {EMBED_DIM} across time",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()

## Summary

This notebook demonstrated how to:

1. **Fetch multi-temporal Sentinel-2 data** from a public STAC catalog using stackstac
2. **Prepare datacubes** with z-score normalization, wavelengths, and cyclic time/location encodings
3. **Run the Clay v1.5 encoder** to produce `[B, 1+1024, 1024]` embeddings
4. **Rearrange patch tokens** back to a `[B, D, 32, 32]` spatial grid for visualization
5. **Inspect individual embedding dimensions** as spatial feature maps -- each dimension captures
   a different learned aspect of the scene (edges, spectral properties, textures)
6. **Track temporal changes** by comparing the same embedding dimension across dates

Key takeaways:
- The CLS token (position 0) is a single 1024-d vector summarizing the entire image
- The remaining 1024 patch tokens form a 32x32 spatial grid, each with 1024 dimensions
- Different dimensions respond to different scene properties, making them useful as
  feature inputs for downstream tasks (classification, change detection, similarity search)

See `wall-to-wall.ipynb` for PCA analysis and fine-tuning a classifier on top of these embeddings.